# Sample Demo of our Project

Our project is an interface that allows for the user to select paper and summary sentences to view evidence retrieval and metrics, but for the one minute demo, we settled for using hardcoded results that were extracted from our project that simulate what a user would see. We chose one paper from our subset of data for this demonstration

## Installations and Imports

In [43]:
import json
from pathlib import Path
from IPython.display import display, Markdown
import csv
import re

## Sample Run on one paper

### Displaying Paper Info

In [44]:

paper_id = "253098895"
paper = json.loads(Path(f"sample_data/{paper_id}.json").read_text(encoding="utf-8"))
title = paper.get("title")
authors = paper.get("authors", [])

sections = paper.get("sections")

chars_used = 0
text_list = []
max_chars = 800

for section in sections:
    text = section.get("text")
    text = " ".join(text.split())

    chars_left = max_chars - chars_used
    if (chars_left <= 0):
        break
    if len(text) > chars_left:
        text = text[:chars_left] + "..."
    text_list.append(text)
    chars_used += len(text)

display(Markdown(f"**Title:** {title}"))
display(Markdown(f"**Authors:** {', '.join(authors)}"))
display(Markdown(f"**Snippet:**"))
display(Markdown("\n\n".join(text_list)))



**Title:** SpaBERT: A Pretrained Language Model from Geographic Data for Geo-Entity Representation

**Authors:** Zekun Li, Jina Kim, Yao-Yi Chiang, Muhao Chen

**Snippet:**

Interpreting human behaviors requires considering human activities and their surrounding environment. Looking at a stopping location, [ Speedway , ], 1 from a person's trajectory, we might assume that this person needs to use the location's amenities if Speedway implies a gas station, and is near a highway exit. We might predict a meetup at [ Speedway , ] if the trajectory travels through many other locations, , , ..., of the same 1 A geographic entity name Speedway and its location (e.g., latitude and longitude). Best viewed in color. name, Speedway , to arrive at [ Speedway , ] in the middle of farmlands. As humans, we are able to make such inferences using the name of a geographic entity (geo-entity) and other entities in a spatial neighborhood. Specifically, we contextualize a geo-enti...

### Displaying Summaries

In [45]:
textrank_summary = Path(f"sample_data/{paper_id}_textrank_summary.txt").read_text(encoding="utf-8")
bart_summary = Path(f"sample_data/{paper_id}_bart_summary.txt").read_text(encoding="utf-8")

display(Markdown("## TextRank Summary"))
display(Markdown(textrank_summary))

display(Markdown("## BART Summary"))
display(Markdown(bart_summary))

## TextRank Summary

Specifically, SPABERT linearizes the 2-dimensional spatial context by forming pseudo sentences that consist of names of the pivot and neighboring geo-entities, ordered by their spatial distance to the pivot. 1 is constructed as: The pseudo sentence starts with the pivot name followed by the names of the pivot's neighboring geo-entities, ordered by their spatial distance to the pivot in ascending order. One task is the masked language modeling (MLM) (Devlin et al., 2019), for which SPABERT needs to learn how to complete the full names of geo-entities from pseudo sentences with randomly masked subtokens using the remaining subtokens and their spatial coordinates (i.e., partial names and spatial relations between subtokens). For a query set Q = {(q i , SC(q i ))} |Q| i=1 , the goal is to find the corresponding geo-entity for each

Here, Q and C are the USGS and Wikidata geo-entities ( §3.1). SPABERT also encodes the spatial relations between the pivot and neighboring geo-entities with a continuous spatial coordinate embedding.

## BART Summary

SPABERT is a LM built upon a pretrained BERT and trained to produce contextualized geo-entity representations given large geographic datasets. For a pivot, p, SPAberT first linearizes its neighboring geo-entitie names to form a BERT-compatible input sequence, called a pseudo sentence. The representations support various downstream applications, including contextualized Geo-entity classification. SPABERT performs the best for the 1:125K maps (30-CA) (Tab. 4) Our hypothesis is that the test maps contain denser geo-entities than other scales. We simulate omission using the OSM dataset by gradually removing random neighbors of a pivot entity.

## Evidence Retrieval Results

To just show an example, we will display the evidence retrieval using the top 3 evidences from the research paper that match the first summary sentence generated with BART. These will not be displayed in any paritcular order, but on our main app.py, we score each one and display them in descending order.

Our full project allows the user to select any of the sentences generated by TextRank or BART and retrieve the top three evidence for each displayed in descending order.

### Selected Sentence
SPABERT is a LM built upon a pretrained BERT and trained to produce contextualized geo-entity representations given large geographic datasets.

In [46]:
faiss_results = json.loads(Path(f"sample_data/{paper_id}_faiss_retrieval_1.json").read_text(encoding="utf-8"))

display(Markdown("## FAISS Retrieval Results"))
for section, snippets in faiss_results.items():
    for snippet in snippets:
        display(Markdown(f"**Section: {section}**"))
        display(Markdown(snippet))

## FAISS Retrieval Results

**Section: Conclusion**

This paper presented SPABERT ( ), a language model trained on geographic datasets for contextualizing geo-entities.

**Section: Introduction**

SPABERT is a LM built upon a pretrained BERT and further trained to produce contextualized geo-entity representations given large geographic datasets.

**Section: Introduction**

To tackle these challenges, we present SPABERT ( ), a LM that captures the spatially varying semantics of geo-entity names using large geographic datasets for entity representation.

## Overall Summary Metrics

In [47]:
def load_metrics(paper_id, model_key):
    metrics_path = Path(f"sample_data/{paper_id}_metrics.tsv")
    with metrics_path.open("r", encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            if row.get("model") == model_key:
                return row

for model_key in ["textrank", "bart"]:
    row = load_metrics(paper_id, model_key)
    model_name = "TextRank" if model_key == "textrank" else "BART"
    display(Markdown(f"## {model_name} Metrics"))
    display(Markdown(f"Flesch Reading Ease: {float(row['flesch_reading_ease']):.1f}"))
    display(Markdown(f"Flesch-Kincaid Grade: {float(row['flesch_kincaid_grade']):.1f}"))
    display(Markdown(f"Grammar/style issues: {row['total_errors']} total ({float(row['error_rate'])*100:.1f} per 100 words)"))
    display(Markdown(f"Fluency (GPT-2 perplexity): {float(row['perplexity']):.1f}"))

## TextRank Metrics

Flesch Reading Ease: 46.4

Flesch-Kincaid Grade: 12.7

Grammar/style issues: 8 total (6.3 per 100 words)

Fluency (GPT-2 perplexity): 48.3

## BART Metrics

Flesch Reading Ease: 32.5

Flesch-Kincaid Grade: 12.5

Grammar/style issues: 4 total (4.4 per 100 words)

Fluency (GPT-2 perplexity): 206.1

In [48]:
results_text = Path(f"sample_data/{paper_id}_results.txt").read_text(encoding="utf-8")
display(Markdown("## ROUGE & NLI Results"))
display(Markdown(f"```\n{results_text}\n```"))

nli_results = Path(f"sample_data/{paper_id}_bart_results_1.txt").read_text(encoding="utf-8")
display(Markdown("## NLI Results"))
display(Markdown(f"```\n{nli_results}\n```"))

## ROUGE & NLI Results

```
ROUGE RESULTS

TextRank ROUGE (v.s. abstract):
rouge1: precision=0.83  recall=0.14  f1=0.25
rouge2: precision=0.55  recall=0.09  f1=0.16
rougeL: precision=0.46  recall=0.08  f1=0.14

BART ROUGE (v.s. abstract):
rouge1: precision=0.79  recall=0.08  f1=0.14
rouge2: precision=0.24  recall=0.02  f1=0.04
rougeL: precision=0.44  recall=0.04  f1=0.08

```

## NLI Results

```
BART x NLI RESULTS

Sentence: SPABERT is a LM built upon a pretrained BERT and trained to produce contextualized geo-entity representations given large geographic datasets.

Section Title: Conclusion
Retrieved Sentence: This paper presented SPABERT ( ), a language model trained on geographic datasets for contextualizing geo-entities.
Entailment Score: 0.0013754687970504165
Neutral Score: 0.9983038902282715
Contradiction Score: 0.0003205907123629004

Section Title: Introduction
Retrieved Sentence: SPABERT is a LM built upon a pretrained BERT and further trained to produce contextualized geo-entity representations given large geographic datasets.
Entailment Score: 0.98780357837677
Neutral Score: 0.011062652803957462
Contradiction Score: 0.0011337385512888432

Section Title: Introduction
Retrieved Sentence: To tackle these challenges, we present SPABERT ( ), a LM that captures the spatially varying semantics of geo-entity names using large geographic datasets for entity representation.
Entailment Score: 0.00048275201697833836
Neutral Score: 0.9993391633033752
Contradiction Score: 0.00017801692592911422

Section Title: Preliminary
Retrieved Sentence: Linearizing Neighboring Geo-entity Names For a pivot, p, SPABERT first linearizes its neighboring geo-entitie names to form a BERT-compatible input sequence, called a pseudo sentence.
Entailment Score: 0.0003072245162911713
Neutral Score: 0.9983951449394226
Contradiction Score: 0.001297594397328794

Section Title: Contextualizing Geo-entities
Retrieved Sentence: Also, SPABERT incorporates a spatial coordinate embedding mechanism, which seeks to represent the spatial relations between the pivot and its neighboring geo-entities.
Entailment Score: 0.00024136477441061288
Neutral Score: 0.9994675517082214
Contradiction Score: 0.00029110844479873776

Section Title: Contextualizing Geo-entities
Retrieved Sentence: Then SPABERT averages the pivot's token-level embeddings to produce a fixedlength embedding for the pivot's contextualized representation.
Entailment Score: 0.00026433554012328386
Neutral Score: 0.9991549253463745
Contradiction Score: 0.0005807921988889575

Section Title: Unknown
Retrieved Sentence: The representations support various downstream applications, including contextualized geo-entity classification and similarity-based inference tasks, such as geo-entity typing and linking.
Entailment Score: 0.0003216483455616981
Neutral Score: 0.9992274045944214
Contradiction Score: 0.00045092060463503003

Section Title: Introduction
Retrieved Sentence: Specifically, we contextualize a geo-entity by a reasonable surrounding neighborhood learned from experience and, from the neighborhood, relate other relevant geo-entities based on their name and spatial relations (e.g., distance) to the geo-entity.
Entailment Score: 0.0005518602556549013
Neutral Score: 0.9982531666755676
Contradiction Score: 0.0011949907056987286

Section Title: Introduction
Retrieved Sentence: This section first presents the preliminary ( §2.1) and then describes the overall approach for learning geo-entities' contextualized representations ( §2.2), the pretraining strategies ( §2.3), and inference procedures ( §2.4).
Entailment Score: 0.0007255052914842963
Neutral Score: 0.9956343770027161
Contradiction Score: 0.0036401653196662664

Section Title: Model
Retrieved Sentence: Here SPABERT performs the best for the 1:125K maps (30-CA) (Tab.
Entailment Score: 0.0005250826943665743
Neutral Score: 0.9990405440330505
Contradiction Score: 0.00043432958773337305

Section Title: Ethical Consideration
Retrieved Sentence: Replacing the backbone of SPABERT with a multi-lingual model and training SPABERT with diverse regions could mitigate the bias.
Entailment Score: 0.0007797214202582836
Neutral Score: 0.9932312965393066
Contradiction Score: 0.005989033728837967

Section Title: Spatial Context and Ablation Study
Retrieved Sentence: Our hypothesis is that the 1:125K maps contain denser geo-entities than the test maps at other scales.
Entailment Score: 0.0009412488434463739
Neutral Score: 0.9957667589187622
Contradiction Score: 0.0032920069061219692

Section Title: Spatial Context and Ablation Study
Retrieved Sentence: Our hypothesis is that the 1:125K maps contain denser geo-entities than the test maps at other scales.
Entailment Score: 0.0009412488434463739
Neutral Score: 0.9957667589187622
Contradiction Score: 0.0032920069061219692

Section Title: Experimental Setup
Retrieved Sentence: Note that the geocoordinates of geo-entities in the scanned maps are unknown.
Entailment Score: 0.0011090446496382356
Neutral Score: 0.995662271976471
Contradiction Score: 0.0032286278437823057

Section Title: Model
Retrieved Sentence: Our hypothesis is that the 1:125K maps contain denser geo-entities than the test maps at other scales.
Entailment Score: 0.0009412488434463739
Neutral Score: 0.9957667589187622
Contradiction Score: 0.0032920069061219692

Section Title: Spatial Context and Ablation Study
Retrieved Sentence: Since the entity omission criteria of the USGS maps are unknown and Wikidata mostly contain important landmark geo-entities, we also simulate omission using the OSM dataset by gradually removing random neighbors of a pivot entity within a fixed neighborhood.
Entailment Score: 0.0030606805812567472
Neutral Score: 0.9826914668083191
Contradiction Score: 0.014247828163206577

Section Title: Pretraining
Retrieved Sentence: Since OSM is crowd-sourced, we clean the raw data by removing non-alphanumeric place names and geo-entities that do not have a place name or geocoordinates.
Entailment Score: 0.0009550440008752048
Neutral Score: 0.6909494400024414
Contradiction Score: 0.3080954849720001

Section Title: Pretraining
Retrieved Sentence: Therefore, we propose and incorporate a masked entity prediction (MEP) task, which randomly masks all subtokens of an entity name in a pseudo sentence.
Entailment Score: 0.0009283627150580287
Neutral Score: 0.9693738222122192
Contradiction Score: 0.02969786338508129


```